In [ ]:
!pip install openai Pillow requests

In [61]:
import os
import requests
import json
from io import BytesIO
from PIL import Image
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()

# Cliente para TEXTO (Gemini es muy estable)
client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.getenv("OPENROUTER_API_KEY"),
)

MODELO_TEXTO = "openrouter/elephant-alpha" 
MODELO_IMAGEN = "sourceful/riverflow-v2-pro"

In [62]:
def generar_leyenda(prompt_historia):
    try:
        respuesta = client.chat.completions.create(
            model=MODELO_TEXTO,
            messages=[
                {"role": "system", "content": "Eres un narrador de fantasia. Escribe una frase breve."},
                {"role": "user", "content": f"Escribe una leyenda sobre: {prompt_historia}"}
            ]
        )
        return respuesta.choices[0].message.content.strip()
    except Exception as e:
        # Si Gemini falla, intentamos un modelo de respaldo rapido
        print(f"Error en Gemini, intentando respaldo: {e}")
        return "Una leyenda grabada en las cenizas del tiempo."

In [63]:
def generar_imagen_robusta(prompt_imagen, nombre_archivo):
    url = "https://openrouter.ai/api/v1/images/generations"
    headers = {
        "Authorization": f"Bearer {os.getenv('OPENROUTER_API_KEY')}",
        "Content-Type": "application/json",
        "HTTP-Referer": "http://localhost:3000", # Requerido por algunos proveedores de OpenRouter
        "X-Title": "Generador Local"
    }
    payload = {
        "model": MODELO_IMAGEN,
        "prompt": prompt_imagen,
        "size": "1024x1024",
        "n": 1
    }

    try:
        print(f"Enviando peticion a {MODELO_IMAGEN}...")
        response = requests.post(url, headers=headers, data=json.dumps(payload), timeout=120)
        
        # Imprimimos el estado para saber que pasa
        print(f"Estado de la respuesta: {response.status_code}")
        
        if response.status_code != 200:
            print(f"Error detectado: {response.text}")
            return None

        resultado = response.json()
        
        if "data" in resultado and len(resultado["data"]) > 0:
            url_imagen = resultado["data"][0]["url"]
            print(f"URL recibida: {url_imagen[:50]}...")
            
            img_response = requests.get(url_imagen)
            if img_response.status_code == 200:
                img_data = img_response.content
                with open(nombre_archivo, 'wb') as f:
                    f.write(img_data)
                return Image.open(BytesIO(img_data))
        else:
            print(f"La API no devolvio datos de imagen: {resultado}")
            return None
            
    except Exception as e:
        print(f"Excepcion durante la generacion: {str(e)}")
        return None

In [64]:
def sintetizar_actualizacion_de_prompt(prompt_base, peticion_cambio):
    """
    Fusiona el historial con la nueva peticion del usuario.
    """
    instrucciones = (
        "Eres un experto en prompts para IA de imagenes. Tu tarea es recibir un prompt actual "
        "y una solicitud de modificacion para crear un nuevo prompt detallado en ingles. "
        "No des explicaciones, devuelve solo el texto del nuevo prompt."
    )
    
    contenido = f"Prompt actual: {prompt_base}\nCambio deseado: {peticion_cambio}"

    try:
        respuesta = client.chat.completions.create(
            model=MODELO_TEXTO,
            messages=[
                {"role": "system", "content": instrucciones},
                {"role": "user", "content": contenido}
            ]
        )
        # Limpieza de comillas
        nuevo_prompt = respuesta.choices[0].message.content.strip().replace('"', '')
        return nuevo_prompt
    except Exception as e:
        print(f"Error al sintetizar el prompt: {e}")
        return prompt_base

In [65]:
def iniciar_generador_conversacional():
    print("--- Generador de Imagenes Interactivo ---")
    personaje = input("Introduce el personaje principal: ")
    escenario = input("Introduce el escenario: ")
    
    # Estado inicial
    prompt_actual = f"Digital painting of {personaje} in {escenario}, highly detailed, cinematic lighting."
    leyenda = generar_leyenda(f"{personaje} en {escenario}")
    version = 1
    
    while True:
        archivo = f"imagen_v{version}.png"
        print(f"\n--- Generando Version {version} ---")
        print(f"Historia: {leyenda}")
        
        imagen = generar_imagen_robusta(prompt_actual, archivo)
        
        if imagen:
            # En un notebook, esto muestra la imagen
            from IPython.display import display
            display(imagen)
        
        feedback = input("\n¿Que cambios quieres hacer? (o escribe 'salir'): ")
        
        if feedback.lower() in ['salir', 'no', 'exit']:
            print("Proceso finalizado.")
            break
            
        # Actualizacion de prompt basandose en el historial
        print("Sintetizando cambios...")
        prompt_actual = sintetizar_actualizacion_de_prompt(prompt_actual, feedback)
        version += 1

# Ejecucion
if __name__ == "__main__":
    iniciar_generador_conversacional()

--- Generador de Imagenes Interactivo ---

--- Generando Version 1 ---
Historia: En el dorado desierto, un ratón devoró un queso ancestral y susurró secretos que hicieron brotar oasis de sueños.
Enviando peticion a sourceful/riverflow-v2-pro...
Estado de la respuesta: 404
Error detectado: <!DOCTYPE html><html lang="en"><head><meta charSet="utf-8"/><meta name="viewport" content="width=device-width, initial-scale=1, minimum-scale=1"/><link rel="stylesheet" href="/_next/static/css/cda3af16020f96ba.css" data-precedence="next"/><link rel="stylesheet" href="/_next/static/css/23565049b3597e0b.css" data-precedence="next"/><link rel="stylesheet" href="/_next/static/css/6d57887cdd55b4e9.css" data-precedence="next"/><link rel="stylesheet" href="/_next/static/css/bdee4bec1afee7ff.css" data-precedence="next"/><link rel="preload" as="script" fetchPriority="low" href="/_next/static/chunks/webpack-db8c3b419101d6da.js"/><script src="/_next/static/chunks/9ba4270a-e6f67529ac26615d.js" async=""></script><

In [66]:
"""
Resumen de flujo:
Usuario da personaje + escenario
        ↓
Prompt → Gemini (TEXT + IMAGE)
        ↓
Se muestra leyenda + portada_inicial.png
        ↓
Usuario da feedback ("hazlo más oscuro", "añade lluvia"...)
        ↓
Prompt original + feedback → Gemini (TEXT + IMAGE)
        ↓
Se muestra leyenda mejorada + portada_final.png
"""

'\nResumen de flujo:\nUsuario da personaje + escenario\n        ↓\nPrompt → Gemini (TEXT + IMAGE)\n        ↓\nSe muestra leyenda + portada_inicial.png\n        ↓\nUsuario da feedback ("hazlo más oscuro", "añade lluvia"...)\n        ↓\nPrompt original + feedback → Gemini (TEXT + IMAGE)\n        ↓\nSe muestra leyenda mejorada + portada_final.png\n'